# Przykłady jakościowe - rysunki rankingów

Składa rysunki z przykładami działania systemu: dla każdego zapytania ranking BAZY i ranking wariantu, po trzy klatki na każdy pokazany fragment. Czyta gotowe katalogi przebiegów z `results/runs/` - nie uruchamia wyszukiwania i nie ładuje żadnego modelu. Jedyna operacja na nagraniach to zdekodowanie klatek, wykonywane raz i zapisywane w pamięci podręcznej.

Rysunek niesie kadry oraz przy każdym wierszu pozycję, nazwę nagrania i przedział czasowy.

**Wymaga:** przebiegów `E2-A` i wariantu każdego przykładu na części testowej, pamięci podręcznej segmentacji, przedziałów czarnego obrazu oraz nagrań w `data/processed/`. Nagrania potrzebne są tylko przy pierwszym uruchomieniu.

**Zapisuje:** `results/reports/examples/candidates_*.csv` i `selection.csv` (wersjonowane, ścieżka audytu wyboru), `data/cache/thumbnails/` (miniatury) oraz `results/figures/examples/*.png` (rysunki, poza kontrolą wersji).

## Komórka konfiguracji

Importy, ustawienia przebiegu i geometria rysunku. Nic tu nie liczy.

Klasa każdego przykładu - znacznik, złożoność, wariant, kierunek - stoi w `CASES` w `src/evaluation/examples.py`. **Wybrane zapytania zapisywane są ręcznie w `selection.csv`**: jeden wiersz na przykład, z `desc_id` wybranego zapytania i kolumną `top`, która mówi, ile pozycji rankingu ma pokazać rysunek. Blok niżej wypisuje wybór razem z pozycjami trafienia po obu stronach.

In [ ]:
import importlib
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation import examples, sheets, thumbnails

for module in (examples, thumbnails, sheets):
    importlib.reload(module)      # the kernel keeps a once-imported module in memory

SPLIT = "test"
#: None: every example is drawn as deep as its own `top` column in selection.csv;
#: a number forces one depth on all of them
ROWS = None
#: True also writes every row as a figure of its own, for captioning one frame
STRIPS = False

print(f"wybor przykladow ({examples.selection_file().name}):")
examples.selection_report()
print()
for key, value in sheets.geometry(examples.TOPK).items():
    print(f"  {key:<20} {value}")

## 1. Kandydaci i plik wyboru

**Zapisuje:** `results/reports/examples/candidates_<przykład>.csv` - cała populacja każdej klasy zapytań, posortowana według przydatności do rysunku. To ona pozwala napisać, spośród ilu zapytań przykład został wybrany, i sprawdzić, które to były.

Plik `selection.csv` powstaje osobno i **nie jest tu nadpisywany** - to w nim stoi wybór autorki. Po ręcznej zmianie `desc_id` albo `top` kolumny pochodne (pozycje trafienia, widoczność w pokazanym top) uzgadnia z przebiegami `examples.write_selection()`.

In [ ]:
written = examples.write_candidates(SPLIT)
print()
print(f"plikow: {len(written)}  ->  {examples.reports_dir().relative_to(ROOT)}")

## 2. Złożenie przykładów

Ranking obu konfiguracji z katalogów przebiegów, zbiór poprawnych fragmentów z `relevance.csv` tego samego przebiegu, przedziały czasowe z pamięci podręcznej segmentacji. Przypadek bez przebiegu wypisuje, czego brakuje, i nie zatrzymuje pozostałych.

Ostatnia linia każdego przypadku to **gotowe zdanie komentarza** - jedno zdanie o tym, co dodanie komponentu zrobiło z pozycją poprawnego fragmentu, razem z dopiskiem o pozycji poza pokazanym top, gdy trafienie tam wypada.

In [ ]:
cases = {}
for spec, pick in examples.selected(SPLIT):
    try:
        cases[spec.stem] = examples.build(spec, pick, SPLIT)
    except (FileNotFoundError, KeyError, ValueError) as problem:
        print(f"{spec.stem}: {problem}")

for stem, case in cases.items():
    print(f"{stem:<9} {case.dataset:<7} top-{case.pick.top}  "
          f"{case.reference.rank_first_correct:>4} -> "
          f"{case.candidate.rank_first_correct:<4} "
          f"|Rel|={case.n_relevant}  populacja {case.candidates_total:>4}")
    print(f"   {case.query.desc}")
    print(f"   {case.comment}")

## 3. Miniatury

Trzy klatki na fragment, w środkach trzech równych części jego **treści** - nie pliku. Fragment może nieść maskę przejścia albo ciemny kadr, a te sekundy nie liczą się jako treść; próbkowanie liniowe po pliku wpadałoby w animację przejścia i rysunek wyglądałby na błąd systemu. Na rysunku nie ma podpisów kolumn - skąd są klatki, wyjaśnia się raz w opisie.

Pierwsze uruchomienie dekoduje klatki z nagrań i trwa kilkanaście sekund; kolejne czytają pamięć podręczną.

**Zapisuje:** `data/cache/thumbnails/<zbiór>/<fragment>_<n>.jpg`.

In [ ]:
images = {}
for stem, case in cases.items():
    images[stem] = thumbnails.ensure(case.dataset, case.fragments)

total = sum(1 for group in images.values() for paths in group.values()
            for path in paths if path.exists())
print()
print(f"klatek w pamieci podrecznej: {total}")

## 4. Rysunki

Dwa pliki na przykład: `<przykład>_baza.png` i `<przykład>_wariant.png` - po jednym na konfigurację, żeby między nimi dało się wstawić nagłówek w rodzaju „BAZA + tożsamość". Każdy jest samą siatką: bez marginesów, bez treści zapytania, bez podpisów kolumn i bez numeru strony. Rysowane w milimetrach, wszystkie o tej samej szerokości 160 mm, z kadrem 43,87 × 24,68 mm.

Ile pozycji pokazuje rysunek, mówi kolumna `top` w `selection.csv`, osobno dla każdego przykładu.

Poprawny fragment oznacza jedna rzecz: słowo `POPRAWNE` w rynience, pogrubione i zielone. Ramki wszystkich kadrów są jednakowe, bo kolorowa obwódka na kadrze z serialu konkuruje z obrazem, zamiast na niego wskazywać; pogrubienie przenosi oznaczenie do druku szarego.

**Zapisuje:** `results/figures/examples/*.png`, nadpisywane przy każdym uruchomieniu. Katalog wyników jest w `.gitignore`: rysunki są złożone z klatek nagrań, które nie podlegają redystrybucji, a repozytorium jest publiczne.

In [ ]:
figures = {}
for stem, case in cases.items():
    figures[stem] = sheets.save(case, images[stem], rows=ROWS, strips=STRIPS)

print()
print(f"plikow: {sum(len(paths) for paths in figures.values())}  ->  "
      f"{sheets.output_dir().relative_to(ROOT)}")

## 5. Podgląd

Wyświetla oba rysunki jednego przykładu, żeby dało się zobaczyć wynik bez otwierania katalogu. Zmień `STEM` na inny przypadek.

In [ ]:
from IPython.display import Image as Preview, display

STEM = "face"

case = cases[STEM]
print(case.query.desc)
print(case.meta_line)
print()
print(case.reference.heading)
display(Preview(filename=str(sheets.output_dir() / f"{STEM}_baza.png"), width=740))
print(case.candidate.heading)
display(Preview(filename=str(sheets.output_dir() / f"{STEM}_wariant.png"), width=740))
print()
print(case.comment)